# End-to-End ML Pipeline: Tesla Sales/Price Data
This notebook covers preprocessing, EDA, feature engineering, regression modeling, hyperparameter tuning, and time series forecasting.

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor

## 2. Data Loading & Initial Inspection

In [ ]:
df = pd.read_csv('tesla_deliveries_dataset_2015_2025.csv')
display(df.head())


## 3. Preprocessing
Create a datetime column from Year and Month, sort the data, and handle any missing values.

In [ ]:

df['Date'] = pd.to_datetime(df[['Year', 'Month']].assign(DAY=1))
# sort
df = df.sort_values('Date').reset_index(drop=True)
#deleting duplicates
df = df.drop_duplicates()
# fillin missing values
df = df.ffill()
display(df[['Date', 'Year', 'Month', 'Estimated_Deliveries', 'Avg_Price_USD']].head())

## 4. Exploratory Data Analysis (EDA)
Visualize distributions, trends over time, and correlations.

In [ ]:
fig, axes =  plt.subplots(2, 2, figsize=(16, 12))

# Trend of Deliveries over time
monthly_deliveries = df.groupby('Date')['Estimated_Deliveries'].sum().reset_index()
sns.lineplot(data=monthly_deliveries, x='Date', y='Estimated_Deliveries', ax=axes[0,0], marker='o')
axes[0,0].set_title('Total Estimated Deliveries Over Time', fontsize=12, fontweight='bold')
axes[0,0].set_ylabel('Deliveries')

# Average Price by Model
sns.boxplot(data=df, x='Model', y='Avg_Price_USD', ax=axes[0,1])
axes[0,1].set_title('Price Distribution by Model', fontsize=12, fontweight='bold')
axes[0,1].set_ylabel('Average Price (USD)')

# Histogram
sns.histplot(data=df, x='Estimated_Deliveries', bins=20, ax=axes[1,0], kde=True)
axes[1,0].set_title('Distribution of Estimated Deliveries', fontsize=12, fontweight='bold')
axes[1,0].set_xlabel('Estimated Deliveries')
axes[1,0].set_ylabel('Frequency')

# Correlation heatmap
axes[1,1].remove()
ax_corr = fig.add_subplot(2, 2, 4)
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', ax=ax_corr, cbar_kws={'label': 'Correlation'})
ax_corr.set_title('Correlation Heatmap', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()


## 5. Feature Engineering
Prepare features for regression modeling. We'll encode categorical variables and scale numerical variables.

In [ ]:
# Define features and target
# Let's predict 'Estimated_Deliveries' based on other factors
target = 'Estimated_Deliveries'

# Drop Date (handled implicitly by Year/Month if needed, or we keep Year/Month)
X = df.drop(columns=[target, 'Date'])
y = df[target]

categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = X.select_dtypes(include=[np.number]).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)

# Create a preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ])

# Fit and transform features
X_processed = preprocessor.fit_transform(X)

# Get feature names
cat_feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols)
feature_names = numerical_cols + list(cat_feature_names)

X_processed_df = pd.DataFrame(X_processed, columns=feature_names)
display(X_processed_df.head())

## 6. Regression Modeling
Train Linear, Ridge, and Lasso regression models and compare performance.

In [ ]:
# Train-test split (80-20)
X_train, X_test, y_train, y_test = train_test_split(X_processed_df, y, test_size=0.2, random_state=42)

# Initialize models
linear_model = LinearRegression()
ridge_model = Ridge(alpha=1.0, random_state=42)
lasso_model = Lasso(alpha=1.0, random_state=42)

# Train all models
linear_model.fit(X_train, y_train)
ridge_model.fit(X_train, y_train)
lasso_model.fit(X_train, y_train)
# Make predictions
y_pred_linear = linear_model.predict(X_test)
y_pred_ridge = ridge_model.predict(X_test)
y_pred_lasso = lasso_model.predict(X_test)

# Evaluation metrics
models = ['Linear Regression', 'Ridge Regression', 'Lasso Regression']
predictions = [y_pred_linear, y_pred_ridge, y_pred_lasso]
rmse_scores = []
mae_scores = []
r2_scores = []

for model_name, y_pred in zip(models, predictions):
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    rmse_scores.append(rmse)
    mae_scores.append(mae)
    r2_scores.append(r2)
    
    print(f"\n{model_name}:")
    print(f"  RMSE: {rmse:.2f}")
    print(f"  MAE:  {mae:.2f}")
    print(f"  R² Score: {r2:.4f}")


fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# RMSE Comparison
axes[0].bar(models, rmse_scores, color=['blue', 'green', 'red'], alpha=0.7)
axes[0].set_ylabel('RMSE', fontsize=11, fontweight='bold')
axes[0].set_title('RMSE Comparison', fontsize=12, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)
for i, v in enumerate(rmse_scores):
    axes[0].text(i, v + 5, f'{v:.2f}', ha='center', fontweight='bold')

# MAE Comparison
axes[1].bar(models, mae_scores, color=['blue', 'green', 'red'], alpha=0.7)
axes[1].set_ylabel('MAE', fontsize=11, fontweight='bold')
axes[1].set_title('MAE Comparison', fontsize=12, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)
for i, v in enumerate(mae_scores):
    axes[1].text(i, v + 5, f'{v:.2f}', ha='center', fontweight='bold')

# R² Score Comparison
axes[2].bar(models, r2_scores, color=['blue', 'green', 'red'], alpha=0.7)
axes[2].set_ylabel('R² Score', fontsize=11, fontweight='bold')
axes[2].set_title('R² Score Comparison', fontsize=12, fontweight='bold')
axes[2].tick_params(axis='x', rotation=45)
for i, v in enumerate(r2_scores):
    axes[2].text(i, v - 0.01, f'{v:.4f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 7. Hyperparameter Tuning
Tune Ridge and Lasso regularization parameters using cross-validation.

In [ ]:
# Define alpha values to test
alphas = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]

ridge_cv_scores = []
lasso_cv_scores = []

print("\nTuning Regularization Parameters:")
print("="*50)

for alpha in alphas:
    ridge = Ridge(alpha=alpha, random_state=42)
    lasso = Lasso(alpha=alpha, random_state=42, max_iter=5000)
    
    # Cross-validation scores (5-fold)
    ridge_scores = cross_val_score(ridge, X_train, y_train, cv=5, scoring='r2')
    lasso_scores = cross_val_score(lasso, X_train, y_train, cv=5, scoring='r2')
    
    ridge_cv_scores.append(ridge_scores.mean())
    lasso_cv_scores.append(lasso_scores.mean())
    
    print(f"Alpha = {alpha:7.3f} | Ridge R²: {ridge_scores.mean():.4f} | Lasso R²: {lasso_scores.mean():.4f}")

# Find best alpha values
best_ridge_alpha = alphas[np.argmax(ridge_cv_scores)]
best_lasso_alpha = alphas[np.argmax(lasso_cv_scores)]

print(f"\nBest Ridge Alpha: {best_ridge_alpha}")
print(f"Best Lasso Alpha: {best_lasso_alpha}")

# Train models with best alpha values
best_ridge = Ridge(alpha=best_ridge_alpha, random_state=42)
best_lasso = Lasso(alpha=best_lasso_alpha, random_state=42, max_iter=5000)

best_ridge.fit(X_train, y_train)
best_lasso.fit(X_train, y_train)

# Predictions with tuned models
y_pred_ridge_tuned = best_ridge.predict(X_test)
y_pred_lasso_tuned = best_lasso.predict(X_test)

# Evaluate tuned models
print("\n" + "="*70)
print("TUNED MODELS PERFORMANCE")
print("="*70)
print(f"\nTuned Ridge (alpha={best_ridge_alpha}):")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_ridge_tuned)):.2f}")
print(f"  MAE:  {mean_absolute_error(y_test, y_pred_ridge_tuned):.2f}")
print(f"  R² Score: {r2_score(y_test, y_pred_ridge_tuned):.4f}")

print(f"\nTuned Lasso (alpha={best_lasso_alpha}):")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_lasso_tuned)):.2f}")
print(f"  MAE:  {mean_absolute_error(y_test, y_pred_lasso_tuned):.2f}")
print(f"  R² Score: {r2_score(y_test, y_pred_lasso_tuned):.4f}")
print("="*70)

# Visualize alpha tuning
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(alphas, ridge_cv_scores, marker='o', linewidth=2, markersize=8, color='green', label='Ridge')
axes[0].axvline(best_ridge_alpha, color='green', linestyle='--', alpha=0.7, label=f'Best: {best_ridge_alpha}')
axes[0].set_xlabel('Alpha', fontweight='bold')
axes[0].set_ylabel('Cross-Validation R² Score', fontweight='bold')
axes[0].set_title('Ridge: Alpha Tuning', fontweight='bold')
axes[0].set_xscale('log')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(alphas, lasso_cv_scores, marker='s', linewidth=2, markersize=8, color='red', label='Lasso')
axes[1].axvline(best_lasso_alpha, color='red', linestyle='--', alpha=0.7, label=f'Best: {best_lasso_alpha}')
axes[1].set_xlabel('Alpha', fontweight='bold')
axes[1].set_ylabel('Cross-Validation R² Score', fontweight='bold')
axes[1].set_title('Lasso: Alpha Tuning', fontweight='bold')
axes[1].set_xscale('log')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Time Series Forecasting
Forecast future total monthly deliveries using Time Series Cross Validation.

In [ ]:
# Aggregate total deliveries by Month
ts_data = df.groupby('Date')['Estimated_Deliveries'].sum().reset_index()
ts_data.set_index('Date', inplace=True)
ts_data = ts_data.asfreq('MS') # Monthly start frequency

# Create features for time series regression
ts_data['Month'] = ts_data.index.month
ts_data['Year'] = ts_data.index.year
ts_data['Lag_1'] = ts_data['Estimated_Deliveries'].shift(1)
ts_data['Lag_2'] = ts_data['Estimated_Deliveries'].shift(2)
ts_data['Lag_12'] = ts_data['Estimated_Deliveries'].shift(12)
ts_data = ts_data.dropna()
X = ts_data[['Month', 'Year', 'Lag_1', 'Lag_2', 'Lag_12']]
y = ts_data['Estimated_Deliveries']
# Train-Test Split for Time Series (keep last 12 months for testing)
X_train_ts = X.iloc[:-12]
y_train_ts = y.iloc[:-12]
X_test_ts = X.iloc[-12:]
y_test_ts = y.iloc[-12:]
# Fit Model with Time Series Cross Validation
tscv = TimeSeriesSplit(n_splits=3)
model = RandomForestRegressor(n_estimators=100, random_state=42)
cv_scores = []
for train_idx, val_idx in tscv.split(X_train_ts):
    X_train_cv, X_val_cv = X_train_ts.iloc[train_idx], X_train_ts.iloc[val_idx]
    y_train_cv, y_val_cv = y_train_ts.iloc[train_idx], y_train_ts.iloc[val_idx]
    model.fit(X_train_cv, y_train_cv)
    preds = model.predict(X_val_cv)
    cv_scores.append(np.sqrt(mean_squared_error(y_val_cv, preds)))

print(f"Time Series CV RMSE scores: {cv_scores}")
print(f"Mean CV RMSE: {np.mean(cv_scores):.2f}\n")

# Train on full train set and forecast
model.fit(X_train_ts, y_train_ts)
forecast = model.predict(X_test_ts)
# Plot the forecast
plt.figure(figsize=(14, 6))
plt.plot(y_train_ts.index, y_train_ts, label='Training Data')
plt.plot(y_test_ts.index, y_test_ts, label='Actual Test Data')
plt.plot(y_test_ts.index, forecast, color='red', label='Forecast')
plt.title('Time Series Cross Validation Forecast: Total Monthly Deliveries')
plt.xlabel('Date')
plt.ylabel('Total Estimated Deliveries')
plt.legend()
plt.show()

# Evaluate Time Series Model
ts_rmse = np.sqrt(mean_squared_error(y_test_ts, forecast))
print(f"Time Series Forecast RMSE: {ts_rmse:.2f}")
